# 🎓 Strathmore University — AI Career Coach
## Module: Fine-Tuning the Dense Career Retriever (RAG Pipeline)
**Student:** Kimathi John Mwiti (169900)  
**Supervisor:** Madam Julliet Kirui  
**Degree:** BSc in Informatics and Computer Science — Capstone Project  

---

### 📌 Notebook Objective
In a **Retrieval-Augmented Generation (RAG)** system, answering career guidance questions accurately requires retrieving verified, domain-specific resources before calling an LLM.

In this notebook, we:
1. **Prepare a domain-specific dataset** of university career advisory question–passage pairs (CV standards, STAR interview methods, local tech career paths).
2. **Fine-tune** the `sentence-transformers/all-MiniLM-L6-v2` embedding model using **Multiple Negatives Ranking Loss (MNRL)**.
3. **Quantitatively evaluate** retrieval performance against the frozen baseline using **Recall@1**, **Recall@3**, and **Mean Reciprocal Rank (MRR)**.
4. **Export evaluation plots** for Chapter 5 of the project report.
5. **Package and download** the trained model artifact for deployment in the mobile app backend.

### Step 1: Install Dependencies
We install `sentence-transformers`, `torch`, `datasets`, `pandas`, `matplotlib`, and `accelerate`.

In [ ]:
# Install required libraries
!pip install -q sentence-transformers datasets accelerate pandas numpy matplotlib

import torch
print("PyTorch Version:", torch.__version__)
print("CUDA Available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("Device Name:", torch.cuda.get_device_name(0))

### Step 2: Knowledge Base & Career Dataset Preparation
We define the verified career knowledge corpus and a comprehensive set of training & evaluation pairs.
*(You can also load a custom CSV dataset from Kaggle or Google Drive using the optional cell below).*

In [ ]:
import pandas as pd
import json

# 1. Verified Career Guidance Knowledge Base Corpus
career_corpus = [
    {
        "id": "cv_layout_01",
        "title": "Software Engineer CV Template",
        "text": "A strong internship or graduate CV for computer science students is strictly one page. Lead with education (course, expected graduation, relevant modules), then 2 to 3 technical projects with impact, followed by technical skills and work experience."
    },
    {
        "id": "cv_bullets_02",
        "title": "CV Bullet Writing & Impact",
        "text": "Write CV bullet points following the formula: Action Verb + Technology/Tool + Measurable Result. Example: 'Built a REST API in Python and FastAPI that cut manual report generation time from 3 hours to 15 minutes.' Avoid generic job duty lists."
    },
    {
        "id": "cv_ats_03",
        "title": "ATS Formatting Rules",
        "text": "For Applicant Tracking Systems (ATS) used by tech recruiters in Kenya, use a clean single-column layout, standard headings (Education, Projects, Skills), standard fonts (Arial, Calibri), and save as a searchable PDF without images, icons, or text boxes."
    },
    {
        "id": "cover_letter_01",
        "title": "Cover Letter Structure",
        "text": "A tech cover letter should be concise (under 300 words). Paragraph 1 states the role applied for. Paragraph 2 connects a specific academic project to the company's business problem. Paragraph 3 details availability and an interview request."
    },
    {
        "id": "interview_star_01",
        "title": "STAR Method for Behavioural Interviews",
        "text": "In behavioural interviews, structure answers using STAR: Situation (set the context), Task (what was required), Action (specific steps YOU took), and Result (measurable outcome). Keep responses under two minutes."
    },
    {
        "id": "interview_questions_02",
        "title": "Common Behavioural Interview Questions",
        "text": "Prepare STAR stories covering 5 core themes: teamwork in group projects, resolving technical conflict, learning from a project failure, demonstrating leadership, and meeting a tight delivery deadline."
    },
    {
        "id": "interview_technical_01",
        "title": "Coding Interview Preparation Strategy",
        "text": "Technical coding screens test arrays, hash maps, strings, and graph traversal. Always speak out loud: restate the problem, walk through an example test case, state time/space complexity, and write clean, readable code before optimizing."
    },
    {
        "id": "career_ds_roles_01",
        "title": "Careers in Data Science & Machine Learning",
        "text": "Data Analyst, Data Scientist, and Machine Learning Engineer are distinct roles. Analysts clean data, build SQL queries, and present BI dashboards. ML Engineers write production code, deploy models via APIs, and monitor pipeline drift."
    },
    {
        "id": "career_entry_ai_02",
        "title": "Landing an Entry-Level AI Role in Kenya",
        "text": "A master's degree is not strictly required for entry-level analyst or junior ML roles in Nairobi. Candidates with Python proficiency, SQL fluency, Git version control, and 1 to 2 deployed portfolio projects are strongly preferred."
    },
    {
        "id": "portfolio_github_01",
        "title": "Building a Technical Portfolio",
        "text": "Feature 2 to 3 well-documented projects rather than 10 incomplete repositories. Each project must have a clear README explaining the problem, architecture diagram, tech stack, and a live demo link hosted on GitHub Pages or cloud."
    },
    {
        "id": "internship_safari_01",
        "title": "Local Internship Programs & Timelines",
        "text": "Graduate and intern recruitment pipelines at regional employers (e.g. Safaricom, Equity Group, Microsoft ADC, local tech hubs) open recruitment cycles in Q1 and Q3. Early preparation of portfolios and algorithmic problem solving is critical."
    },
    {
        "id": "networking_linkedin_01",
        "title": "Professional LinkedIn Optimization",
        "text": "Optimize your LinkedIn headline to reflect your aspirational role and core technologies (e.g. 'CS Student @ Strathmore | Aspiring ML Engineer | Python, PyTorch, SQL'). Engage with engineering blogs and alumni networks."
    }
]

# 2. Training Question-Passage Pairs
train_data = [
    {"query": "How many pages should an internship CV be?", "positive": "A strong internship or graduate CV for computer science students is strictly one page."},
    {"query": "How should I structure my tech resume for internships?", "positive": "Lead with education (course, expected graduation, relevant modules), then 2 to 3 technical projects with impact, followed by technical skills and work experience."},
    {"query": "How do I write strong resume bullet points?", "positive": "Write CV bullet points following the formula: Action Verb + Technology/Tool + Measurable Result."},
    {"query": "What layout is best for passing ATS resume screening?", "positive": "use a clean single-column layout, standard headings (Education, Projects, Skills), standard fonts (Arial, Calibri), and save as a searchable PDF."},
    {"query": "How long should my software cover letter be?", "positive": "A tech cover letter should be concise (under 300 words)."},
    {"query": "What is the STAR technique in job interviews?", "positive": "In behavioural interviews, structure answers using STAR: Situation (set the context), Task (what was required), Action (specific steps YOU took), and Result."},
    {"query": "How should I answer behavioural interview questions?", "positive": "Prepare STAR stories covering 5 core themes: teamwork in group projects, resolving technical conflict, learning from a project failure."},
    {"query": "What topics appear most in coding interviews?", "positive": "Technical coding screens test arrays, hash maps, strings, and graph traversal."},
    {"query": "What should I do during a technical coding interview?", "positive": "Always speak out loud: restate the problem, walk through an example test case, state time/space complexity, and write clean, readable code."},
    {"query": "What is the difference between a Data Analyst and ML Engineer?", "positive": "Analysts clean data, build SQL queries, and present BI dashboards. ML Engineers write production code, deploy models via APIs."},
    {"query": "Do I need a masters degree to get an AI job in Nairobi?", "positive": "A master's degree is not strictly required for entry-level analyst or junior ML roles in Nairobi. Candidates with Python proficiency, SQL fluency, Git version control."},
    {"query": "How many projects should I show on my portfolio website?", "positive": "Feature 2 to 3 well-documented projects rather than 10 incomplete repositories. Each project must have a clear README."},
    {"query": "Where should I host my software projects portfolio?", "positive": "Each project must have a clear README explaining the problem, architecture diagram, tech stack, and a live demo link hosted on GitHub Pages or cloud."},
    {"query": "When do tech companies in Kenya hire interns?", "positive": "Graduate and intern recruitment pipelines at regional employers (e.g. Safaricom, Equity Group, Microsoft ADC) open recruitment cycles in Q1 and Q3."},
    {"query": "How do I optimize my LinkedIn profile for tech jobs?", "positive": "Optimize your LinkedIn headline to reflect your aspirational role and core technologies (e.g. 'CS Student @ Strathmore | Aspiring ML Engineer')."}
]

# 3. Held-out Evaluation Set (for testing Recall@K)
eval_data = [
    {"query": "How should a computer science student format their resume?", "target_id": "cv_layout_01"},
    {"query": "What formula makes a good CV accomplishment bullet?", "target_id": "cv_bullets_02"},
    {"query": "How can I make sure my resume passes automated ATS filters?", "target_id": "cv_ats_03"},
    {"query": "How to structure an interview answer about teamwork?", "target_id": "interview_star_01"},
    {"query": "How should I talk through a programming screen?", "target_id": "interview_technical_01"},
    {"query": "What entry level data roles are realistic for new grads?", "target_id": "career_entry_ai_02"},
    {"query": "How should I showcase my GitHub projects to recruiters?", "target_id": "portfolio_github_01"},
    {"query": "Which Kenyan tech companies offer graduate trainee schemes?", "target_id": "internship_safari_01"}
]

print(f"Knowledge Corpus passages: {len(career_corpus)}")
print(f"Training pairs: {len(train_data)}")
print(f"Evaluation pairs: {len(eval_data)}")

### (Optional) Load Custom Kaggle / External Dataset
If you have an external CSV dataset from Kaggle or Google Drive, you can easily load it by uncommenting the cell below:

In [ ]:
# OPTIONAL: Load your custom CSV file from Kaggle or local upload
# from google.colab import files
# uploaded = files.upload() # upload your 'career_qa.csv'
# custom_df = pd.read_csv('career_qa.csv')
# print(custom_df.head())

### Step 3: Model Setup & Multiple Negatives Ranking Loss (MNRL)
We load the pre-trained `all-MiniLM-L6-v2` model from Hugging Face.
**Loss Function**: Multiple Negatives Ranking Loss (MNRL) pulls the query vector and its positive passage vector close together in the shared embedding space, while pushing away all other passages in the mini-batch as in-batch negatives.

In [ ]:
from sentence_transformers import SentenceTransformer, InputExample, losses
from torch.utils.data import DataLoader

# Load base sentence embedding model
model_name = "sentence-transformers/all-MiniLM-L6-v2"
print(f"Loading base model: {model_name}...")
model = SentenceTransformer(model_name)

# Create training InputExamples
train_examples = [
    InputExample(texts=[item["query"], item["positive"]])
    for item in train_data
]

# Configure DataLoader with batch size
train_dataloader = DataLoader(train_examples, shuffle=True, batch_size=4)

# Define Multiple Negatives Ranking Loss
train_loss = losses.MultipleNegativesRankingLoss(model)

print(f"Prepared {len(train_examples)} training pairs. Ready for fine-tuning.")

### Step 4: Fine-Tune the Retriever Model
We train the model for 4 epochs with warm-up steps.

In [ ]:
import os

output_model_dir = "./strathmore_career_retriever"
os.makedirs(output_model_dir, exist_ok=True)

print("Starting retriever fine-tuning...")
model.fit(
    train_objectives=[(train_dataloader, train_loss)],
    epochs=4,
    warmup_steps=5,
    show_progress_bar=True,
    output_path=output_model_dir
)
print(f"\n✅ Model fine-tuning complete! Saved to '{output_model_dir}'")

### Step 5: Quantitative Evaluation (Recall@1, Recall@3, and MRR)
We compare the **Baseline MiniLM** against the **Fine-Tuned Career Retriever** on our held-out evaluation set.

In [ ]:
import numpy as np

# Load baseline and fine-tuned models
baseline_model = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")
finetuned_model = SentenceTransformer(output_model_dir)

corpus_texts = [doc["text"] for doc in career_corpus]
corpus_ids = [doc["id"] for doc in career_corpus]

def evaluate_model(eval_model, k_values=[1, 3]):
    # Pre-encode knowledge corpus
    doc_embeddings = eval_model.encode(corpus_texts, convert_to_numpy=True, normalize_embeddings=True)
    
    recall_scores = {k: 0 for k in k_values}
    mrr_sum = 0.0

    for item in eval_data:
        q_emb = eval_model.encode([item["query"]], convert_to_numpy=True, normalize_embeddings=True)[0]
        # Cosine similarity via dot product of normalized vectors
        scores = doc_embeddings @ q_emb
        ranked_indices = np.argsort(-scores)
        ranked_ids = [corpus_ids[i] for i in ranked_indices]

        # Check hits
        for k in k_values:
            if item["target_id"] in ranked_ids[:k]:
                recall_scores[k] += 1

        # Mean Reciprocal Rank (MRR)
        if item["target_id"] in ranked_ids:
            rank = ranked_ids.index(item["target_id"]) + 1
            mrr_sum += 1.0 / rank

    total = len(eval_data)
    results = {f"Recall@{k}": round(recall_scores[k] / total, 4) for k in k_values}
    results["MRR"] = round(mrr_sum / total, 4)
    return results

baseline_metrics = evaluate_model(baseline_model)
finetuned_metrics = evaluate_model(finetuned_model)

metrics_df = pd.DataFrame([baseline_metrics, finetuned_metrics], index=["Baseline (all-MiniLM-L6-v2)", "Fine-Tuned Career Retriever"])
print("--- QUANTITATIVE EVALUATION RESULTS ---")
display(metrics_df)

### Step 6: Generate Evaluation Charts for Project Report (Chapter 5)

In [ ]:
import matplotlib.pyplot as plt

labels = ["Recall@1", "Recall@3", "MRR"]
baseline_vals = [baseline_metrics["Recall@1"], baseline_metrics["Recall@3"], baseline_metrics["MRR"]]
finetuned_vals = [finetuned_metrics["Recall@1"], finetuned_metrics["Recall@3"], finetuned_metrics["MRR"]]

x = np.arange(len(labels))
width = 0.35

plt.figure(figsize=(8, 5))
rects1 = plt.bar(x - width/2, baseline_vals, width, label="Baseline (Frozen MiniLM)", color="#5C6BC0")
rects2 = plt.bar(x + width/2, finetuned_vals, width, label="Fine-Tuned Career Retriever", color="#26A69A")

for rect in rects1:
    h = rect.get_height()
    plt.annotate(f"{h:.0%}", xy=(rect.get_x() + rect.get_width()/2, h),
                 xytext=(0, 3), textcoords="offset points", ha='center', va='bottom', fontweight='bold')

for rect in rects2:
    h = rect.get_height()
    plt.annotate(f"{h:.0%}", xy=(rect.get_x() + rect.get_width()/2, h),
                 xytext=(0, 3), textcoords="offset points", ha='center', va='bottom', fontweight='bold')

plt.ylabel("Score", fontsize=12)
plt.title("Career Guidance Retrieval Evaluation (Baseline vs. Fine-Tuned)", fontsize=13, fontweight='bold')
plt.xticks(x, labels, fontsize=11, fontweight='bold')
plt.ylim(0, 1.15)
plt.legend(loc="upper left")
plt.grid(axis='y', linestyle='--', alpha=0.5)
plt.tight_layout()

plt.savefig("retrieval_evaluation_colab.png", dpi=200)
print("Saved chart to retrieval_evaluation_colab.png")
plt.show()

### Step 7: Live Interactive Query Test
Test the fine-tuned model live on custom student queries!

In [ ]:
corpus_embeddings = finetuned_model.encode(corpus_texts, convert_to_numpy=True, normalize_embeddings=True)

def ask_career_retriever(student_query, top_k=2):
    print(f"\n🔎 Student Query: '{student_query}'")
    q_emb = finetuned_model.encode([student_query], convert_to_numpy=True, normalize_embeddings=True)[0]
    scores = corpus_embeddings @ q_emb
    top_idx = np.argsort(-scores)[:top_k]
    
    for rank, idx in enumerate(top_idx, start=1):
        doc = career_corpus[idx]
        print(f"  Rank {rank} [Score: {scores[idx]:.4f}] -> {doc['title']}")
        print(f"    Content: \"{doc['text']}\"\n")

# Run test queries
ask_career_retriever("How should I structure my CV for tech internships in Nairobi?")
ask_career_retriever("What is the STAR method for answering interview questions?")
ask_career_retriever("Do I need a masters degree to get an entry level AI or data role?")

### Step 8: Package & Download the Trained Model
Download your trained model weights (`career_retriever_model.zip`) to use locally in your app backend.

In [ ]:
import shutil
from google.colab import files

# Zip the trained model directory
zip_filename = "strathmore_career_retriever.zip"
shutil.make_archive("strathmore_career_retriever", 'zip', output_model_dir)
print(f"Model compressed into {zip_filename}. Triggering download...")

files.download(zip_filename)
files.download("retrieval_evaluation_colab.png")
print("✅ Download started!")